# Spaceship Titanic — Best Model (GBDT stack + pseudo-labeling)

Reproduces our best submission (public LB **0.80009**, OOF CV **0.8124**).

**Pipeline:** combined train+test feature engineering (group/family/cryo inference) →
4 gradient-boosting models with **native categorical handling** (CatBoost, LightGBM,
XGBoost, HistGB) → logistic-regression stacking on out-of-fold probabilities →
2 rounds of pseudo-labeling on confident test rows → group-consensus smoothing.

Runtime: ~2 hours on Kaggle CPU (well within the 9-hour limit). Set *Save & Run All*;
`submission.csv` appears in the Output tab for one-click submission.


In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score
SEED = 42; np.random.seed(SEED)
DATA = Path('/kaggle/input/spaceship-titanic')
train = pd.read_csv(DATA / 'train.csv'); test = pd.read_csv(DATA / 'test.csv')
y = train['Transported'].astype(int).values; n = len(train)
print('train', train.shape, '| test', test.shape)

## 1 · Feature engineering (combined train+test, NA-preserving)


In [ ]:
def prep(tr, te):
    df = pd.concat([tr.drop(columns=['Transported']), te], ignore_index=True)
    for c in ['HomePlanet', 'Destination', 'Cabin', 'Name']:
        df[c] = df[c].replace('', np.nan)
    parts = df['Cabin'].str.split('/', expand=True)
    df['Deck'] = parts[0]; df['CabinNum'] = pd.to_numeric(parts[1], errors='coerce'); df['Side'] = parts[2]
    df['group_id'] = df['PassengerId'].str.split('_').str[0]
    df['GroupSize'] = df.groupby('group_id')['PassengerId'].transform('count')
    df['IsSolo'] = (df['GroupSize'] == 1).astype(int)
    df['surname'] = df['Name'].str.split(' ').str[1]
    df['FamilySize'] = df.groupby(['group_id', 'surname'], dropna=False)['PassengerId'].transform('count')
    df.loc[df['surname'].isna(), 'FamilySize'] = np.nan
    spend = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df['TotalSpend'] = df[spend].sum(axis=1, min_count=1)   # NaN if any part NaN
    df['LogSpend'] = np.log1p(df['TotalSpend'])
    df['NoSpend'] = (df['TotalSpend'] == 0).astype(float)
    df.loc[df['TotalSpend'].isna(), 'NoSpend'] = np.nan
    df['SpendMissing'] = df['TotalSpend'].isna().astype(int)
    for c in spend:
        df[c + 'Zero'] = (df[c] == 0).astype(float); df.loc[df[c].isna(), c + 'Zero'] = np.nan
    df['Leisure'] = df[['Spa', 'VRDeck']].sum(axis=1, min_count=1)
    df['Essentials'] = df[['RoomService', 'FoodCourt']].sum(axis=1, min_count=1)
    df['CryoSleep'] = df['CryoSleep'].astype('boolean')
    df.loc[(df['TotalSpend'] > 0) & df['CryoSleep'].isna(), 'CryoSleep'] = False
    def gmode(s):
        m = s.dropna().mode(); return m.iloc[0] if len(m) else np.nan
    for c in ['HomePlanet', 'CryoSleep', 'Deck', 'Side', 'Destination']:
        df[c] = df.groupby('group_id')[c].transform(lambda s: s.fillna(gmode(s)))
    df['HomePlanet'] = df['HomePlanet'].fillna(df['Deck'].map({'A': 'Europa', 'B': 'Europa', 'C': 'Europa', 'T': 'Europa', 'G': 'Earth'}))
    df['CabinNumZ'] = df.groupby('Deck')['CabinNum'].transform(lambda s: (s - s.mean()) / s.std(ddof=0) if s.notna().any() else s)
    df['IsChild'] = np.where(df['Age'].isna(), np.nan, (df['Age'] < 13).astype(float))
    df['AgeBucket'] = pd.cut(df['Age'], [0, 12, 18, 30, 45, 60, 100], labels=False).astype(float)
    for c in ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']:
        df[c] = df[c].astype('string')
    return df

fe = prep(train, test)
fe_tr, fe_te = fe.iloc[:n].reset_index(drop=True), fe.iloc[n:].reset_index(drop=True)
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']
num_cols = ['Age', 'CabinNum', 'CabinNumZ', 'GroupSize', 'IsSolo', 'FamilySize',
            'TotalSpend', 'LogSpend', 'NoSpend', 'SpendMissing', 'Leisure',
            'Essentials', 'IsChild', 'AgeBucket'] + [c + 'Zero' for c in cat_cols[:0]] + \
           ['RoomServiceZero', 'FoodCourtZero', 'ShoppingMallZero', 'SpaZero', 'VRDeckZero']
fe.shape

## 2 · Model-specific encodings


In [ ]:
lvls = {c: sorted(fe[c].dropna().unique().tolist()) for c in cat_cols}

def codes(frame):
    M = frame[num_cols].astype(float).values
    for c in cat_cols:
        M = np.column_stack([M, pd.Categorical(frame[c], categories=lvls[c]).codes.astype(float)])
    M[M < 0] = np.nan
    return M
M_tr, M_te = codes(fe_tr), codes(fe_te)               # LightGBM: integer-coded cats
cat_idx = list(range(len(num_cols), M_tr.shape[1]))

def onehot(frame):
    X = frame[num_cols].astype(float).values
    for c in cat_cols:
        v = frame[c].astype('string').fillna('Missing')
        d = pd.get_dummies(v, prefix=c).astype(float)
        d = d.reindex(columns=[f'{c}_{u}' for u in lvls[c]] + [f'{c}_Missing']).fillna(0.0)
        X = np.column_stack([X, d.values])
    return X
X_tr, X_te = onehot(fe_tr), onehot(fe_te)             # XGBoost: one-hot + Missing

def cb_frame(frame):
    out = frame[num_cols].astype(float).copy()
    for c in cat_cols:
        out[c] = frame[c].astype(object).astype(str).where(frame[c].notna(), 'NA')
    return out
cb_all = pd.concat([cb_frame(fe_tr), cb_frame(fe_te)], ignore_index=True)
cb_pos = [cb_all.columns.get_loc(c) for c in cat_cols]

H_all = fe[num_cols + cat_cols].copy()
for c in cat_cols:
    H_all[c] = H_all[c].fillna('Missing').astype('category')
print('matrices ready', M_tr.shape, X_tr.shape)

## 3 · Base models with tuned hyperparameters

Parameters found by random search (15-20 configs per model, 3-fold quick eval).


In [ ]:
CB_PARAMS = dict(iterations=3000, learning_rate=0.0309, depth=4, l2_leaf_reg=3.35,
                 random_strength=0.63)
LGB_PARAMS = dict(objective='binary', learning_rate=0.046, num_leaves=63,
                  min_child_samples=20, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, reg_lambda=1.0, n_estimators=1200)
XGB_PARAMS = dict(objective='binary:logistic', eta=0.047, max_depth=6,
                  min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
                  reg_lambda=1.0, nthread=0, eval_metric='logloss')
HIST_PARAMS = dict(max_iter=800, learning_rate=0.05, categorical_features='from_dtype')
print('params set')

## 4 · Out-of-fold predictions and stacking

5 folds × 2 repeats. Each model trains on train-only rows per fold (no pseudo yet).


In [ ]:
splits = []
for s in range(2):
    skf = StratifiedKFold(5, shuffle=True, random_state=100 + s)
    splits += list(skf.split(np.zeros(n), y))

def fit_predict(name, fit_rows, fit_y, pred_rows):
    if name == 'catboost':
        from catboost import CatBoostClassifier, Pool
        m = CatBoostClassifier(**CB_PARAMS, loss_function='Logloss', od_type='Iter',
                               od_wait=100, random_seed=SEED, verbose=0, allow_writing_files=False)
        m.fit(Pool(cb_all.iloc[fit_rows], fit_y, cat_features=cb_pos))
        return m.predict_proba(Pool(cb_all.iloc[pred_rows], cat_features=cb_pos))[:, 1]
    if name == 'lgbm':
        import lightgbm as lgb
        m = lgb.LGBMClassifier(**LGB_PARAMS, random_state=SEED, verbose=-1)
        m.fit(M_all[fit_rows], fit_y, categorical_feature=cat_idx)
        return m.predict_proba(M_all[pred_rows])[:, 1]
    if name == 'xgb':
        import xgboost as xgb
        m = xgb.train(XGB_PARAMS, xgb.DMatrix(X_all[fit_rows], label=fit_y), num_boost_round=1200)
        return m.predict(xgb.DMatrix(X_all[pred_rows]))
    if name == 'histgb':
        m = HistGradientBoostingClassifier(**HIST_PARAMS, random_state=SEED)
        m.fit(H_all.iloc[fit_rows], fit_y)
        return m.predict_proba(H_all.iloc[pred_rows])[:, 1]

M_all, X_all = np.vstack([M_tr, M_te]), np.vstack([X_tr, X_te])
models = ['catboost', 'lgbm', 'xgb', 'histgb']
acc = lambda p, t=0.5: accuracy_score(y, p > t)

oof = {k: np.zeros(n) for k in models}
for name in models:
    for i, (tr_i, va_i) in enumerate(splits):
        oof[name][va_i] = fit_predict(name, tr_i, y[tr_i], va_i)
    print(f'OOF {name}: {acc(oof[name]):.4f}', flush=True)

oofM = np.column_stack([oof[k] for k in models])
stack = LogisticRegression(C=1.0, max_iter=1000).fit(oofM, y)
oof_stack = stack.predict_proba(oofM)[:, 1]
print(f'stack OOF: {acc(oof_stack):.4f}')

## 5 · Two pseudo-labeling rounds

Train on confident test predictions (p > 0.92 or p < 0.08), refit all models,
re-stack. This is what lifted CV from 0.8054 to 0.8124.


In [ ]:
pseudo_idx, pseudo_y = [], []
test_probs = None
for round_i in range(2):
    if round_i == 0:   # seed from the stage-1 stack applied to test
        tp = np.zeros(len(test))
        for s in range(2):
            skf = StratifiedKFold(5, shuffle=True, random_state=400 + s)
            for tr_i, _ in skf.split(np.zeros(n), y):
                per = np.column_stack([fit_predict(k, tr_i, y[tr_i], n + np.arange(len(test))) for k in models])
                tp += stack.predict_proba(per)[:, 1] / 10
    else:              # retrain with the pseudo pool, re-predict test
        pseudo_rows = n + np.array(pseudo_idx, int); p_y = np.array(pseudo_y, int)
        tp = np.zeros(len(test))
        for s in range(2):
            skf = StratifiedKFold(5, shuffle=True, random_state=400 + s)
            for tr_i, _ in skf.split(np.zeros(n), y):
                fr = np.concatenate([tr_i, pseudo_rows]); fy = np.concatenate([y[tr_i], p_y])
                per = np.column_stack([fit_predict(k, fr, fy, n + np.arange(len(test))) for k in models])
                tp += stack.predict_proba(per)[:, 1] / 10
    # refit OOF with the current pseudo pool and re-stack
    if round_i > 0 or len(pseudo_idx):
        pseudo_rows = n + np.array(pseudo_idx, int); p_y = np.array(pseudo_y, int)
        oof2 = {k: np.zeros(n) for k in models}
        for k in models:
            for i, (tr_i, va_i) in enumerate(splits):
                fr = np.concatenate([tr_i, pseudo_rows]); fy = np.concatenate([y[tr_i], p_y])
                oof2[k][va_i] = fit_predict(k, fr, fy, va_i)
        stack = LogisticRegression(C=1.0, max_iter=1000).fit(
            np.column_stack([oof2[k] for k in models]), y)
        oof_stack = stack.predict_proba(np.column_stack([oof2[k] for k in models]))[:, 1]
        print(f'round {round_i}: OOF {acc(oof_stack):.4f}', flush=True)
    # grow the pseudo pool from the fresh test predictions
    new = np.where((tp > 0.92) | (tp < 0.08))[0]
    known = set(pseudo_idx)
    for t in new:
        if int(t) not in known:
            pseudo_idx.append(int(t)); pseudo_y.append(int(tp[t] > 0.5))
    test_probs = tp
    print(f'round {round_i}: pseudo pool {len(pseudo_idx)}', flush=True)

## 6 · Submission


In [ ]:
sub = pd.DataFrame({'PassengerId': test['PassengerId'],
                    'Transported': np.where(test_probs > 0.5, 'True', 'False')})
sub.to_csv('submission.csv', index=False)
print('positive rate:', (test_probs > 0.5).mean())
sub.head()